# 📗 순공잔디 MLflow 튜토리얼

이 노트북은 팀원들이 MLflow의 전체 흐름을 직접 체험할 수 있도록 만들어졌습니다.

## 전체 흐름
```
1. MLflow 서버 연결 확인
    ↓
2. 모델 학습 + 실험 기록
    ↓
3. UI에서 실험 결과 확인
    ↓
4. 저장된 모델로 예측
    ↓
5. REST API로 예측 호출
```

## 사전 준비
```bash
# 1-1. (선택) 기존에 이미 컨테이너를 한번이라도 띄었다면 -v를 통해 볼륨을 삭제하고 다시 빌드하는것을 권장합니다. (이러면 DB에 있는 내용들도 전부 삭제됨.. 책임은 본인이 ㅎㅎ)
docker compose down -v

# 1-2. Docker 컨테이너가 떠있어야 합니다
docker compose up -d --build

# 2. 가상환경 만든 후 필요한 패키지 설치(ai 디렉토리 안에 requirements_ai.txt가 있습니다.)
pip install -r requirements_ai.txt
```

---
## Step 1. MLflow 서버 연결 확인

MLflow 서버가 정상적으로 떠있는지 확인합니다.  
브라우저에서 **http://localhost:5000** 이 열리면 서버가 준비된 것입니다.

In [ ]:
import requests

MLFLOW_URI = "http://localhost:5000"

try:
    res = requests.get(f"{MLFLOW_URI}/health", timeout=3)
    if res.status_code == 200:
        print("✅ MLflow 서버 연결 성공!")
        print(f"   주소: {MLFLOW_URI}")
    else:
        print(f"⚠️  서버 응답 이상: {res.status_code}")
except Exception as e:
    print("❌ MLflow 서버에 연결할 수 없습니다.")
    print("   → docker compose up -d 실행했는지 확인해주세요")
    print(f"   에러: {e}")

---
## Step 2. 모델 학습 + 실험 기록

MLflow의 핵심 개념 3가지:

| 개념 | 설명 | 비유 |
|---|---|---|
| **Experiment** | 실험 묶음 | GitHub 레포지토리 |
| **Run** | 실험 1회 실행 | Git 커밋 |
| **Artifact** | 모델 파일, 그래프 등 저장물 | Git에서 추적하는 파일 |

아래 셀을 실행하면 MLflow UI에 실험 결과가 기록됩니다.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# MLflow 서버 주소 설정
mlflow.set_tracking_uri(MLFLOW_URI)

# 실험 이름 설정 (UI에서 폴더처럼 보임)
# 같은 이름의 실험이 있으면 거기에 추가됨
mlflow.set_experiment("iris-classification")

print(f"실험 추적 서버: {mlflow.get_tracking_uri()}")
print("실험 이름: iris-classification")

In [ ]:
# 데이터 준비
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target,
    test_size=0.2,
    random_state=42
)

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")
print(f"클래스: {iris.target_names.tolist()}")

In [ ]:
# 실험 실행 — with 블록 안의 내용이 하나의 Run으로 기록됨
with mlflow.start_run(run_name="random-forest-v1"):

    # --- 하이퍼파라미터 설정 ---
    n_estimators = 100
    max_depth = 3

    # --- 모델 학습 ---
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    # --- 평가 ---
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    # --- MLflow에 기록 ---
    # 파라미터: 학습에 사용한 설정값
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # 메트릭: 모델 성능 수치
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)

    # 모델 저장 + 레지스트리 등록
    mlflow.sklearn.log_model(
        model,
        "random-forest-model",
        registered_model_name="iris-classifier"  # Models 탭에 등록
    )

    run_id = mlflow.active_run().info.run_id
    print("✅ 실험 기록 완료!")
    print(f"   Run ID : {run_id}")
    print(f"   accuracy: {accuracy:.4f}")
    print(f"   f1_score: {f1:.4f}")
    print(f"\n👉 UI에서 확인: {MLFLOW_URI}")

``` bash
✅ 실험 기록 완료!
   Run ID : da39ee0d179e43d488bf39fd6cdeaeca
   accuracy: 1.0000
   f1_score: 1.0000
```
위와 같은 결과가 나오면 정상적으로 실행된것입니다. UI로 들어가서도 실행 결과를 확인할 수 있습니다.


---
## Step 3. UI에서 실험 결과 확인

브라우저에서 **http://localhost:5000** 접속 후 아래 순서로 확인하세요.

### Experiments 탭 (실험 기록)
```
왼쪽 사이드바 → iris-classification 클릭
    → random-forest-v1 Run 클릭
        → Parameters 탭: n_estimators, max_depth 확인
        → Metrics 탭: accuracy, f1_score 확인
        → Artifacts 탭: 저장된 모델 파일 확인
```

### Models 탭 (모델 레지스트리)
```
상단 Models 탭 클릭
    → iris-classifier 클릭
        → Version 1 확인
        → Stage 변경 가능: None → Staging → Production
```

---
## Step 4. 저장된 모델로 예측

레지스트리에 등록된 모델을 불러와서 예측합니다.  
모델 파일을 직접 관리할 필요 없이 이름과 버전으로 불러올 수 있습니다.

In [ ]:
import numpy as np

mlflow.set_tracking_uri(MLFLOW_URI)

# 모델 레지스트리에서 로드
# models:/<모델이름>/<버전> 형식
model = mlflow.sklearn.load_model("models:/iris-classifier/1")

# 예측 테스트
labels = ["setosa", "versicolor", "virginica"]

test_cases = [
    ([5.1, 3.5, 1.4, 0.2], "setosa"),
    ([6.0, 2.7, 5.1, 1.6], "versicolor"),
    ([6.9, 3.1, 5.4, 2.1], "virginica"),
]

print("🔍 예측 결과:")
print(f"{'입력값':<30} {'예측':>12} {'정답':>12} {'일치':>6}")
print("-" * 65)
for features, expected in test_cases:
    pred = model.predict(np.array([features]))[0]
    pred_label = labels[pred]
    match = "✅" if pred_label == expected else "❌"
    print(f"{str(features):<30} {pred_label:>12} {expected:>12} {match:>6}")

---
## Step 5. REST API로 예측 호출

실제 Spring Boot 백엔드에서 AI 모델을 호출하는 방식입니다.  

### 서빙 서버 먼저 띄우기
터미널에서 아래 명령 실행 후 이 셀을 실행하세요:
```bash
docker exec -d \
  -e MLFLOW_TRACKING_URI=http://localhost:5000 \
  jandi-mlflow \
  mlflow models serve \
  -m "models:/iris-classifier/1" \
  -h 0.0.0.0 -p 5002 \
  --no-conda --env-manager local

# 서버 뜰 때까지 5초 대기
sleep 5
```

In [ ]:
import requests
import json

SERVING_URI = "http://localhost:5002"

# 서빙 서버 준비 확인
try:
    health = requests.get(f"{SERVING_URI}/health", timeout=3)
    print("✅ 서빙 서버 준비 완료")
except:
    print("❌ 서빙 서버가 아직 안 떴습니다. 위의 터미널 명령을 먼저 실행하세요.")

In [ ]:
# REST API로 예측 요청
# Spring Boot에서 이 방식으로 호출합니다
# (내부 통신은 http://mlflow:5002/invocations)

payload = {
    "inputs": [[5.1, 3.5, 1.4, 0.2]]  # setosa
}

response = requests.post(
    f"{SERVING_URI}/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
)

result = response.json()
labels = ["setosa", "versicolor", "virginica"]

print(f"요청 데이터: {payload['inputs'][0]}")
print(f"API 응답:   {result}")
print(f"예측 클래스: {labels[result['predictions'][0]]}")

---
### 새로운 모델 실험할 때
```python
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("실험-이름")  # 본인 실험 폴더

with mlflow.start_run(run_name="이름"):
    mlflow.log_param("파라미터명", 값)
    mlflow.log_metric("메트릭명", 값)
    mlflow.sklearn.log_model(model, "모델명")
```

### 모델 버전 관리 규칙
| Stage | 의미 | 사용 시점 |
|---|---|---|
| **None** | 실험 중 | 개발/테스트 단계 |
| **Staging** | 검증 중 | 팀 리뷰 요청할 때 |
| **Production** | 서비스 중 | 실제 API에 붙일 때 |

### 자주 쓰는 명령어
```bash
# 전체 인프라 시작
docker compose up -d

# MLflow UI
open http://localhost:5000

# 서빙 서버 띄우기
docker exec -d -e MLFLOW_TRACKING_URI=http://localhost:5000 jandi-mlflow \
  mlflow models serve -m "models:/모델명/버전" -h 0.0.0.0 -p 5002 --no-conda --env-manager local

# MLflow 로그 확인
docker compose logs mlflow -f
```